In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [27]:
movies_df = pd.read_csv("../data/movies_clean.csv")
print(movies_df.shape)
movies_df.head()

(1279, 6)


,Film,Year,Awards,Nominations,Decade,Win_Rate
0,One Battle After Another,2025,6,13,2020,0.461538
1,Sinners,2025,4,16,2020,0.250000
2,Frankenstein,2025,3,9,2020,0.333333
3,KPop Demon Hunters,2025,2,2,2020,1.000000
4,Hamnet,2025,1,8,2020,0.125000


In [28]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

def try_fetch_page(url):
    """Attempt to fetch a Wikipedia page, return the soup if successful."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code == 200:
            return BeautifulSoup(response.text, "lxml")
    except Exception:
        pass
    return None


def extract_reception(soup):
    """Given a page's soup, try to extract the critical response text."""
    target_headings = ["critical response", "critical reception", "reception"]
    for target in target_headings:
        for heading in soup.find_all(["h2", "h3"]):
            if heading.get_text(strip=True).lower() == target:
                paragraphs = []
                for sibling in heading.find_all_next():
                    if sibling.name in ["h2", "h3"]:
                        break
                    if sibling.name == "p":
                        paragraphs.append(sibling.get_text(strip=True))
                combined = " ".join(paragraphs)
                if combined:
                    return combined
    return None


def get_reception_text(film_title, year):
    """Try a few common Wikipedia naming patterns to find a film's reception text."""
    candidates = [
        f"{film_title} ({year} film)",
        f"{film_title} (film)",
        film_title,
    ]
    
    for candidate in candidates:
        page_title = candidate.replace(" ", "_")
        url = f"https://en.wikipedia.org/wiki/{page_title}"
        
        soup = try_fetch_page(url)
        if soup:
            text = extract_reception(soup)
            if text:
                return text
        
        time.sleep(2)
    
    return None

In [29]:
test_text = get_reception_text("Titanic", 1997)
print(test_text[:300] if test_text else "Not found")

Titanicgarnered mostly positive reviews from film critics, and was positively reviewed by audiences and scholars, who commented on its cultural, historical, and political impacts.[208][209][210]On thereview aggregatorwebsiteRotten Tomatoes, it has an approval rating of 88% based on 255 reviews, with


In [ ]:
top30 = movies_df.sort_values('Awards', ascending=False).head(30).copy()

reception_texts = []

for idx, row in top30.iterrows():
    print(f"Fetching: {row['Film']} ({row['Year']})...")
    text = get_reception_text(row['Film'], row['Year'])
    reception_texts.append(text)
    time.sleep(3)

top30['reception_text'] = reception_texts

found = top30['reception_text'].notna().sum()
print(f"\nSuccessfully found reception text for {found} out of {len(top30)} films")

Fetching: Ben-Hur (1959)...
Fetching: The Lord of the Rings: The Return of the King (2003)...
Fetching: Titanic (1997)...
Fetching: West Side Story (1961)...
Fetching: Gigi (1958)...
Fetching: The Last Emperor (1987)...
Fetching: The English Patient (1996)...
Fetching: From Here to Eternity (1953)...


In [ ]:
missing = top30[top30['reception_text'].isna()][['Film', 'Year']]
print(missing)

In [ ]:
sentiment_df = top30.dropna(subset=['reception_text']).copy()
print(f"Working with {len(sentiment_df)} films")
sentiment_df[['Film', 'Year', 'Awards', 'reception_text']].head()

In [ ]:
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    scores = sia.polarity_scores(text)
    return scores['compound']

sentiment_df['sentiment_score'] = sentiment_df['reception_text'].apply(get_sentiment_score)

sentiment_df[['Film', 'Year', 'Awards', 'sentiment_score']].sort_values('sentiment_score', ascending=False)

In [ ]:
cabaret_text = sentiment_df[sentiment_df['Film'] == 'Cabaret']['reception_text'].values[0]
print(cabaret_text[:800])

In [ ]:
correlation = sentiment_df['Awards'].corr(sentiment_df['sentiment_score'])
print(f"Correlation between Awards and Sentiment Score: {correlation:.3f}")

# Also check without the Cabaret outlier, since we now understand why it's skewed
sentiment_df_no_outlier = sentiment_df[sentiment_df['Film'] != 'Cabaret']
correlation_clean = sentiment_df_no_outlier['Awards'].corr(sentiment_df_no_outlier['sentiment_score'])
print(f"Correlation without Cabaret outlier: {correlation_clean:.3f}")

In [ ]:
def classify_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

sentiment_df['sentiment_label'] = sentiment_df['sentiment_score'].apply(classify_sentiment)

sentiment_df[['Film', 'Year', 'Awards', 'sentiment_score', 'sentiment_label']].sort_values('sentiment_score', ascending=False)

In [ ]:
sentiment_df.to_csv("../data/reviews_with_sentiment.csv", index=False)
print(f"Saved {len(sentiment_df)} films with sentiment scores and labels")

## Sentiment Analysis Findings (Connected to Awards Dataset)

- Analyzed critical reception text scraped from Wikipedia for the top 30 most-awarded 
  films in our dataset (27/30 successfully retrieved)
- Used VADER to score each film's reception text from -1 (negative) to +1 (positive), 
  **then classified each score into positive/negative/neutral categories using VADER's 
  standard thresholds (≥0.05 = positive, ≤-0.05 = negative, else neutral)**
- Sentiment scores were overwhelmingly positive (median ~0.98), which makes sense since 
  these are all award-winning, critically significant films — 26 of 27 films classified 
  as positive
- One notable outlier: Cabaret (1972) scored -0.91 (classified negative) despite 
  genuinely positive reviews — VADER misread words like "bleak," "despair," and 
  "cynical" as negative, when critics were actually praising how well the film 
  captured its dark subject matter (Nazi-era Germany). This reveals a real limitation 
  of lexicon-based sentiment tools: they can't distinguish between negative words 
  describing content vs. negative words expressing opinion.
- Correlation between sentiment score and Awards won: essentially zero (r = -0.002; 
  r = 0.115 excluding the Cabaret outlier)
- This suggests critical reception language and Academy Award success are largely 
  independent — winning Oscars depends on factors beyond how positively critics wrote 
  about a film (e.g., campaign strategy, timing, branch voting patterns, industry politics)